# 401 · Protocol Buffers wire format playground

This notebook goes with the article
[Protobuf wire format](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/401/protobuf-wire-format/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/401/wire_format_playground.ipynb)

Library tutorials often hide what actually appears on the wire.
Here you build the basic pieces yourself—tags, variable-length integers, length-delimited fields, nesting, and repeated fields—
and check each step against known hexadecimal examples.

Run the steps **in order**. Later cells reuse functions defined earlier.

> **A note on numbers:** sizes and timings in these notebooks are only illustrations. For measured library comparisons on this project’s harness, use the suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) pages.


## Setup

Only the Python standard library is required. You do not need the `protoc` compiler for this playground.
The next cells define small helpers for printing and checking hexadecimal bytes.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Optional, Tuple


def hex_bytes(b: bytes) -> str:
    return " ".join(f"{x:02x}" for x in b)


def assert_hex(actual: bytes, expected_hex: str, label: str = "") -> None:
    exp = bytes.fromhex(expected_hex.replace(" ", ""))
    assert actual == exp, f"{label}: got {hex_bytes(actual)!r}, expected {hex_bytes(exp)!r}"
    print(f"OK {label}: {hex_bytes(actual)}")



## 1. Encode a key (tag)

Every field on the wire begins with a **key** (also called a tag).
The key packs the field number and the wire type into one integer:

`key = (field_number << 3) | wire_type`

That integer is then written as a variable-length integer (varint).
Field **names** never appear in the binary encoding.

| Wire type name | Numeric value | Payload shape |
|----------------|--------------:|---------------|
| VARINT | 0 | variable-length integer |
| I64 | 1 | eight raw bytes |
| LEN | 2 | length, then that many bytes |
| I32 | 5 | four raw bytes |

When you run the cell, you should see:

- field 1 with VARINT → hex `08`
- field 2 with LEN → hex `12`
- field 16 with VARINT → hex `80 01` (the key itself needs two bytes)


In [ ]:
def encode_varint(u: int) -> bytes:
    if u < 0:
        raise ValueError("unsigned varint only in this lab subset")
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_key(field_number: int, wire_type: int) -> bytes:
    return encode_varint((field_number << 3) | wire_type)


# Field 1 VARINT → 0x08; field 2 LEN → 0x12; field 16 VARINT → 0x80 0x01
assert_hex(encode_key(1, 0), "08", "field1 varint key")
assert_hex(encode_key(2, 2), "12", "field2 len key")
assert_hex(encode_key(16, 0), "80 01", "field16 varint key")



## 2. Encode variable-length integers (base-128)

Integers use a compact encoding: each byte carries seven data bits, and the high bit means “more bytes follow.”
While the value is at least 128, emit `(value & 0x7f) | 0x80` and shift down by seven bits.
The last byte has the high bit cleared.

You should see these examples:

- `1` → `01`
- `127` → `7f`
- `128` → `80 01`
- `300` → `ac 02`


In [ ]:
for n, hx in [(1, "01"), (127, "7f"), (128, "80 01"), (300, "ac 02")]:
    assert_hex(encode_varint(n), hx, f"varint {n}")



## 3. Decode a varint with safety bounds

A decoder must not assume the buffer is well formed.
If the input ends while a varint still claims “more bytes follow,” that is an error.
This teaching decoder also rejects encodings longer than ten bytes (a common cap for 64-bit values).

You should successfully decode `ac 02` as `300`.
A single byte `80` should raise `WireError` because the stream ended too early.


In [ ]:
class WireError(Exception):
    pass


def decode_varint(buf: bytes, i: int = 0) -> Tuple[int, int]:
    value = 0
    shift = 0
    bytes_read = 0
    while True:
        if i >= len(buf):
            raise WireError("truncated varint")
        b = buf[i]
        i += 1
        bytes_read += 1
        if bytes_read > 10:
            raise WireError("overlong varint")
        value |= (b & 0x7F) << shift
        if (b & 0x80) == 0:
            break
        shift += 7
    return value, i


assert decode_varint(bytes.fromhex("ac 02"))[0] == 300
try:
    decode_varint(bytes.fromhex("80"))  # continues, EOF
    raise AssertionError("expected WireError")
except WireError as e:
    print("OK truncated varint:", e)



## 4. Length-delimited string field

Strings, bytes, and nested messages all use wire type LEN (value 2).
On the wire the order is always: key, then the length as a varint, then exactly that many payload bytes.

When encoding, build the payload first so you know the length before you write it.

For `name = "Ada"` on field 2 you should get hex `12 03 41 64 61`
(`12` is the key, `03` is the length, and `41 64 61` is UTF-8 for `Ada`).


In [ ]:
def encode_string_field(field_number: int, s: str) -> bytes:
    payload = s.encode("utf-8")
    return encode_key(field_number, 2) + encode_varint(len(payload)) + payload


# name="Ada" on field 2 → 12 03 41 64 61
assert_hex(encode_string_field(2, "Ada"), "12 03 41 64 61", "string Ada")



## 5. Nested messages and unpacked repeated fields

A nested message is not a special wire type.
It is an ordinary length-delimited field whose payload is itself a complete message encoding.

An **unpacked** repeated field writes one full field (key plus value) for each element.
The same key may appear many times.

You should see:

- manager with `id = 2` → `1a 02 08 02`
- tags `[1, 2]` → `20 01 20 02`


In [ ]:
def encode_uint32_field(field_number: int, v: int) -> bytes:
    if v == 0:
        return b""  # proto3 omit default
    return encode_key(field_number, 0) + encode_varint(v)


# manager={id=2} on field 3 → 1a 02 08 02
inner = encode_uint32_field(1, 2)
nested = encode_key(3, 2) + encode_varint(len(inner)) + inner
assert_hex(nested, "1a 02 08 02", "nested manager id=2")

# tags=[1,2] field 4 unpacked → 20 01 20 02
tags = b"".join(encode_key(4, 0) + encode_varint(t) for t in (1, 2))
assert_hex(tags, "20 01 20 02", "unpacked tags")



## 6. MiniUser golden examples G1–G5 (preview of the full lab)

The rest of the 401 lab uses a tiny teaching message called MiniUser.
It is **not** the large suite schema under `benchmark_v2.proto`.

```protobuf
message MiniUser {
  uint32 id = 1;
  string name = 2;
  MiniUser manager = 3;
  repeated uint32 tags = 4;
}
```

Encode the five golden cases with proto3-style omission of empty defaults.
Your hexadecimal output should match the table in the lab article (Ada, empty message, id 300, tags, nested manager).


In [ ]:
@dataclass
class MiniUser:
    id: int = 0
    name: str = ""
    manager: Optional["MiniUser"] = None
    tags: List[int] = field(default_factory=list)


def encode_mini_user(u: MiniUser) -> bytes:
    out = bytearray()
    out += encode_uint32_field(1, u.id)
    if u.name:
        out += encode_string_field(2, u.name)
    if u.manager is not None:
        inner = encode_mini_user(u.manager)
        out += encode_key(3, 2) + encode_varint(len(inner)) + inner
    for t in u.tags:
        out += encode_key(4, 0) + encode_varint(t)
    return bytes(out)


GOLDENS = {
    "G1": (MiniUser(id=1, name="Ada"), "08 01 12 03 41 64 61"),
    "G2": (MiniUser(), ""),
    "G3": (MiniUser(id=300), "08 ac 02"),
    "G4": (MiniUser(tags=[1, 2]), "20 01 20 02"),
    "G5": (MiniUser(manager=MiniUser(id=2)), "1a 02 08 02"),
}

for label, (user, hx) in GOLDENS.items():
    raw = encode_mini_user(user)
    if hx == "":
        assert raw == b"", label
        print(f"OK {label}: <empty>")
    else:
        assert_hex(raw, hx, label)



## Next

Continue to the full lab, where you also implement decoding, skip unknown fields, and fail closed on truncated input:

- Notebook: [lab_mini_protobuf_encoder.ipynb](./lab_mini_protobuf_encoder.ipynb)
- Article: [Protobuf wire format](../../401/protobuf-wire-format.md)
